# Gemini version of 04b — cosine similarity of utterances to adult mean

Parallel to `04b_talk_generate_cossim_adults`, using `utterance_gemini_embeddings.csv`
from `03g_talk_generate_gemini_embedding`. Both full and half-time utterance
embeddings are already complete in that file (no quota gating needed here, unlike the
drawing branch's half-time section). No column-wise z-scoring — Gemini's embeddings
are already L2-normalized.

Output: `data/preprocessed/gemini/talk_cossim_adults.csv`, same columns as the CLIP
version (`participant_id, age_months, target_category, utterance_full_cossim_adults,
utterance_half_cossim_adults`).

In [1]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd

SEA_ANIMALS = ["crab", "octopus", "seahorse", "shark", "turtle", "whale"]

EMBED_CSV = "../data/embeddings/gemini/utterance_gemini_embeddings.csv"
OUT_CSV = "../data/preprocessed/gemini/talk_cossim_adults.csv"

FLAGS_CSV = "../data/tidy/utterances.csv"  # interference flags

# Interference trials are excluded throughout, so they enter neither the adult mean nor the output
_flags = pd.read_csv(FLAGS_CSV)
INTERFERENCE = set(zip(_flags.loc[_flags["utterance_interference"] == 1, "participant_id"].str.strip(),
                       _flags.loc[_flags["utterance_interference"] == 1, "target_category"].str.strip().str.lower()))

In [2]:
def parse_embedding(x):
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, list):
        return np.array(x, dtype=float)
    if pd.isna(x):
        return np.nan
    return np.array(ast.literal_eval(x), dtype=float)


def average_embedding(embedding_series):
    """Mean of valid embeddings, re-normalized to unit length."""
    valid = [e for e in embedding_series if isinstance(e, np.ndarray) and e.size > 0]
    if not valid:
        return None
    mean = np.mean(np.vstack(valid), axis=0)
    return mean / np.linalg.norm(mean)


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [3]:
utterances = pd.read_csv(EMBED_CSV)
utterances["participant_id"] = utterances["participant_id"].astype(str).str.strip()
utterances["target_category"] = utterances["target_category"].astype(str).str.strip().str.lower()
utterances = utterances[utterances["target_category"].isin(SEA_ANIMALS)].copy()
utterances = utterances[[k not in INTERFERENCE for k in zip(utterances["participant_id"], utterances["target_category"])]].copy()

utterances["utterance_full_embedding"] = utterances["utterance_full_embedding"].apply(parse_embedding)
utterances["utterance_half_embedding"] = utterances["utterance_half_embedding"].apply(parse_embedding)

print(utterances.shape)
print(utterances["is_adult"].value_counts())

(631, 8)
is_adult
False    511
True     120
Name: count, dtype: int64


In [4]:
adult_mask = utterances["is_adult"]
child_mask = ~utterances["is_adult"]

adult_full_by_cat = (
    utterances.loc[adult_mask].groupby("target_category")["utterance_full_embedding"]
    .apply(average_embedding).to_dict()
)
adult_half_by_cat = (
    utterances.loc[adult_mask].groupby("target_category")["utterance_half_embedding"]
    .apply(average_embedding).to_dict()
)

utterances["utterance_full_cossim_adults"] = np.nan
utterances["utterance_half_cossim_adults"] = np.nan


def sim_to_adult(row, col, adult_by_cat):
    emb = row[col]
    proto = adult_by_cat.get(row["target_category"])
    if not isinstance(emb, np.ndarray) or proto is None:
        return np.nan
    return cosine(emb, proto)


utterances.loc[child_mask, "utterance_full_cossim_adults"] = utterances.loc[child_mask].apply(
    sim_to_adult, axis=1, col="utterance_full_embedding", adult_by_cat=adult_full_by_cat
)
utterances.loc[child_mask, "utterance_half_cossim_adults"] = utterances.loc[child_mask].apply(
    sim_to_adult, axis=1, col="utterance_half_embedding", adult_by_cat=adult_half_by_cat
)

print("Adults:", adult_mask.sum(), " Children:", child_mask.sum())
print(utterances.loc[child_mask, ["utterance_full_cossim_adults", "utterance_half_cossim_adults"]].describe().round(3))

Adults: 120  Children: 511
       utterance_full_cossim_adults  utterance_half_cossim_adults
count                       422.000                       422.000
mean                          0.858                         0.835
std                           0.056                         0.057
min                           0.635                         0.638
25%                           0.829                         0.801
50%                           0.874                         0.849
75%                           0.900                         0.876
max                           0.952                         0.941


In [5]:
result = utterances.loc[child_mask, [
    "participant_id", "age_months", "target_category",
    "utterance_full_cossim_adults", "utterance_half_cossim_adults",
]].copy()

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT_CSV, index=False)
print(f"Saved {result.shape[0]} rows -> {OUT_CSV}")

Saved 511 rows -> ../data/preprocessed/gemini/talk_cossim_adults.csv
